# Convert FlyWire Drosophila connectome to libsonata-compatible circuit

Converts the FlyWire v783 connectome data (~138k neurons, ~15M synapses) into
SONATA-format HDF5 node/edge files with a standard circuit config.

**Source:** `../fly-brain/data/` (FlyWire v783 completeness CSV + connectivity parquet)

**Output:** `../../../obi-output/nest_fly_brain/circuit/`

### Original LIF model parameters (Shiu et al.)

| Parameter | Value | Description |
|-----------|-------|-------------|
| v_0 / v_rst | -52 mV | Resting / reset potential |
| v_th | -45 mV | Spike threshold |
| t_mbr | 20 ms | Membrane time constant |
| tau | 5 ms | Synaptic time constant |
| t_rfc | 2.2 ms | Refractory period |
| t_dly | 1.8 ms | Synaptic delay |
| w_syn | 0.275 mV | Base synaptic weight (Brian2 voltage-based) |

The edge `conductance` field stores raw `Excitatory x Connectivity` values
(signed integers: positive = excitatory, negative = inhibitory).
In Brian2 the effective weight is `conductance * 0.275 mV`.

In [ ]:
import json

import h5py
import numpy as np
import pandas as pd
from pathlib import Path

FLY_BRAIN_DIR = Path("/Users/james/Documents/obi/code/obi-main/fly-brain")
COMPLETENESS_CSV = FLY_BRAIN_DIR / "data" / "2025_Completeness_783.csv"
CONNECTIVITY_PARQUET = FLY_BRAIN_DIR / "data" / "2025_Connectivity_783.parquet"
OUTPUT_DIR = Path("../../../obi-output/circuit/nest_fly_brain")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_comp = pd.read_csv(COMPLETENESS_CSV, index_col=0)
df_con = pd.read_parquet(CONNECTIVITY_PARQUET)

n_neurons = len(df_comp)
n_edges = len(df_con)
n_exc = (df_con["Excitatory"] == 1).sum()
n_inh = (df_con["Excitatory"] == -1).sum()

print(f"Neurons: {n_neurons:,}")
print(f"Synapses: {n_edges:,} ({n_exc:,} excitatory, {n_inh:,} inhibitory)")

### Create SONATA node and edge HDF5 files

In [ ]:
# --- Nodes ---
nodes_path = OUTPUT_DIR / "fly_nodes.h5"
with h5py.File(nodes_path, "w") as f:
    pop = f.create_group("nodes/fly")
    pop.create_dataset("node_id", data=np.arange(n_neurons, dtype=np.uint64))
    pop.create_dataset("node_type_id", data=np.zeros(n_neurons, dtype=np.uint64))
    pop.create_dataset("node_group_id", data=np.zeros(n_neurons, dtype=np.uint64))
    pop.create_dataset("node_group_index", data=np.arange(n_neurons, dtype=np.uint64))
    grp = pop.create_group("0")
    grp.create_dataset("flywire_id", data=df_comp.index.values.astype(np.int64))

print(f"Nodes written to {nodes_path} ({nodes_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# --- Edges (sorted by target for SONATA convention) ---
edges_path = OUTPUT_DIR / "fly_fly_edges.h5"

sort_idx = df_con["Postsynaptic_Index"].values.argsort(kind="mergesort")
src_ids = df_con["Presynaptic_Index"].values[sort_idx]
tgt_ids = df_con["Postsynaptic_Index"].values[sort_idx]
conductance = df_con["Excitatory x Connectivity"].values[sort_idx].astype(np.float64)

with h5py.File(edges_path, "w") as f:
    pop = f.create_group("edges/fly_to_fly")
    pop.create_dataset("source_node_id", data=src_ids.astype(np.uint64))
    pop.create_dataset("target_node_id", data=tgt_ids.astype(np.uint64))
    pop.create_dataset("edge_type_id", data=np.zeros(n_edges, dtype=np.uint32))
    pop.create_dataset("edge_group_id", data=np.zeros(n_edges, dtype=np.uint16))
    pop.create_dataset("edge_group_index", data=np.arange(n_edges, dtype=np.uint32))
    grp = pop.create_group("0")
    grp.create_dataset("conductance", data=conductance)
    grp.create_dataset("delay", data=np.full(n_edges, 1.8, dtype=np.float64))

print(f"Edges written to {edges_path} ({edges_path.stat().st_size / 1e6:.1f} MB)")

### Create circuit config and node sets

In [ ]:
node_sets = {"All": {"population": "fly"}}
node_sets_path = OUTPUT_DIR / "node_sets.json"
with node_sets_path.open("w", encoding="utf-8") as f:
    json.dump(node_sets, f, indent=2)

circuit_config = {
    "manifest": {"$BASE_DIR": "."},
    "node_sets_file": str(node_sets_path.absolute()),
    "networks": {
        "nodes": [
            {
                "nodes_file": str(nodes_path.absolute()),
                "populations": {"fly": {"type": "point_process"}},
            },
        ],
        "edges": [
            {
                "edges_file": str(edges_path.absolute()),
                "populations": {"fly_to_fly": {"type": "chemical"}},
            },
        ],
    },
}

circuit_config_path = OUTPUT_DIR / "circuit_config.json"
with circuit_config_path.open("w", encoding="utf-8") as f:
    json.dump(circuit_config, f, indent=2)

print(f"Circuit config: {circuit_config_path}")
print(f"Node sets: {node_sets_path}")

In [ ]:
# Verify
with h5py.File(nodes_path, "r") as f:
    n = f["nodes/fly/node_type_id"].shape[0]
    print(f"Node population 'fly': {n:,} neurons")

with h5py.File(edges_path, "r") as f:
    e = f["edges/fly_to_fly/source_node_id"].shape[0]
    w = f["edges/fly_to_fly/0/conductance"]
    print(f"Edge population 'fly_to_fly': {e:,} edges")
    print(f"  conductance range: [{w[:].min():.0f}, {w[:].max():.0f}]")
    print(f"  delay: {f['edges/fly_to_fly/0/delay'][0]:.1f} ms (constant)")